In [ ]:
import argparse
import numpy as np
from PIL import Image

## Defining the cluster class with attributes k and max itr
class K_means:
    def __init__(self, k=2, max_iter=100, distance='euclidean'):
        self.k = k
        self.max_iter = max_iter

        if distance == 'euclidean':
            self.distance_func = lambda a, b: np.linalg.norm(a - b)
        elif distance == 'manhattan':
            self.distance_func = lambda a, b: np.sum(np.abs(a - b))
        elif distance == 'maximum':
            self.distance_func = lambda a, b: np.max(np.abs(a - b))
        else:
            raise ValueError("distance must be euclidean, manhattan, or maximum")
## getting the data in a class method
    def fit(self, data):
        self.data = data
        self.n_samples, self.n_features = data.shape
        self.clustering()
## Defining the k-means class
    def clustering(self):
        first_points = np.random.choice(self.n_samples, self.k, replace=False) ## Choosing random k- first points from the data
        centroids = [self.data[i] for i in first_points]

        for _ in range(self.max_iter):
            clusters = {i: [] for i in range(self.k)}

            for point in self.data:
                distances = [(self.distance_func(point, c), idx)
                             for idx, c in enumerate(centroids)]
                label = min(distances)[1]
                clusters[label].append(point)
## Calculates the mean of the points in the same cluster and assumes them as new centriods
            new_centroids = {}
            for i, pts in clusters.items():
                new_centroids[i] = np.mean(np.array(pts), axis=0)

            centroids = [new_centroids[i] for i in range(self.k)]

        self.centroids = centroids
        self.clusters = clusters
## Predicts the labels of the cluster
    def predict(self, X):
        labels = []
        for point in X:
            distances = [self.distance_func(point, c) for c in self.centroids]
            labels.append(np.argmin(distances))
        return np.array(labels)






In [ ]:
## Results of the clustering on iris only the atrributes are taken into account and not the labels.

import numpy as np


from sklearn.datasets import load_iris

iris = load_iris()
X = iris.data
y = iris.target


kmeans = K_means(k=3, distance='euclidean')
kmeans.fit(X)

labels_kmeans = kmeans.predict(X)

print("K-means cluster labels:")
print(labels_kmeans)


K-means cluster labels:
[1 0 0 0 1 1 0 1 0 0 1 0 0 0 1 1 1 1 1 1 1 1 0 1 1 0 1 1 1 0 0 1 1 1 0 0 1
 1 0 1 1 0 0 1 1 0 1 0 1 0 2 2 2 2 2 2 2 1 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2
 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 1 2 2 2 2 1 2 2 2 2 2 2 2 2 2 2 2 2
 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2
 2 2]


In [ ]:
### Command line interface
def main():
    parser = argparse.ArgumentParser(description="K-means image segmentation")
    parser.add_argument("--image", required=True, help="Path to input image")
    parser.add_argument("--k", type=int, default=3)
    parser.add_argument("--distance", default="euclidean",
                        choices=["euclidean", "manhattan", "maximum"])
    parser.add_argument("--output", default="kmeans_output.png")

    args = parser.parse_args()

    img = Image.open(args.image)
    img_np = np.array(img)
    h, w, c = img_np.shape

    pixels = img_np.reshape(-1, 3).astype(float)

    model = K_means(k=args.k, distance=args.distance)
    model.fit(pixels)

    labels = model.predict(pixels)

    segmented = np.zeros_like(pixels)
    for i, c in enumerate(model.centroids):
        segmented[labels == i] = c

    segmented = segmented.reshape(h, w, 3).astype(np.uint8)
    Image.fromarray(segmented).save(args.output)


if __name__ == "__main__":
    main()

## Example iuput: python kmeans.py --image flower.png --k 5 --distance euclidean
